In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# 🍊 Trabajo Final IPDI: Detección y Estimación de Producción de Naranjas\n",
    "## Sistema de Entrenamiento Distribuido (GitHub + Drive)\n",
    "\n",
    "**Flujo de Trabajo:**\n",
    "1. **Código:** Se descarga fresco desde GitHub (rama `rama_leom` o la que uses).\n",
    "2. **Datos:** Se inyectan desde Google Drive (`dataset.zip`) al disco local de Colab.\n",
    "3. **Ejecución:** Se entrena en GPU T4 y los resultados se guardan en Drive.\n",
    "\n",
    "---"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### 1. Configuración de Credenciales y Rutas\n",
    "Define aquí dónde están tus cosas."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# --- USUARIO: CONFIGURA ESTO UNA VEZ ---\n",
    "\n",
    "# 1. URL de tu repositorio GitHub\n",
    "REPO_URL = \"https://github.com/LeonEspinosa/IPDI_TrabajoFinal_G10_YOLO.git\"\n",
    "BRANCH = \"rama_leom\"  # La rama donde estás trabajando\n",
    "\n",
    "# 2. Ruta de tu dataset en Google Drive\n",
    "# Ubicación: Mi unidad > IPDI > Trabajo Final > dataset.zip\n",
    "DRIVE_DATASET_PATH = \"/content/drive/MyDrive/IPDI/Trabajo Final/dataset.zip\"\n",
    "\n",
    "# 3. Carpeta para guardar los resultados del entrenamiento\n",
    "# Los guardaremos en una subcarpeta 'Resultados_YOLO' dentro de tu carpeta de trabajo\n",
    "DRIVE_OUTPUT_DIR = \"/content/drive/MyDrive/IPDI/Trabajo Final/Resultados_YOLO\"\n",
    "\n",
    "print(\"✅ Configuración cargada. Apuntando a 'Trabajo Final'.\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### 2. Montaje y Clonación\n",
    "Conectamos los discos duros y traemos el cerebro (código)."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from google.colab import drive\n",
    "import os\n",
    "import shutil\n",
    "\n",
    "# 1. Montar Drive\n",
    "drive.mount('/content/drive')\n",
    "\n",
    "# 2. Clonar/Actualizar Repositorio\n",
    "REPO_NAME = REPO_URL.split(\"/\")[-1].replace(\".git\", \"\")\n",
    "CODE_DIR = f\"/content/{REPO_NAME}\"\n",
    "\n",
    "if os.path.exists(CODE_DIR):\n",
    "    print(\"🔄 Repositorio detectado. Actualizando código (git pull)...\")\n",
    "    %cd {CODE_DIR}\n",
    "    !git checkout {BRANCH}\n",
    "    !git pull\n",
    "else:\n",
    "    print(\"⬇️ Clonando repositorio desde cero...\")\n",
    "    %cd /content\n",
    "    !git clone -b {BRANCH} {REPO_URL}\n",
    "    %cd {CODE_DIR}\n",
    "\n",
    "print(f\"✅ Código listo en: {os.getcwd()}\")\n",
    "\n",
    "# 3. Instalación de Dependencias del Repo\n",
    "!pip install -r requirements.txt\n",
    "!pip install ultralytics  # Asegurar que YOLO esté instalado"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### 3. Ingesta de Datos (Data Pipeline)\n",
    "Traemos el zip de Drive y lo descomprimimos en el entorno local efímero (/content/dataset) para máxima velocidad de lectura."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import yaml\n",
    "\n",
    "LOCAL_DATA_DIR = \"/content/dataset_naranjas\"\n",
    "\n",
    "def prepare_data():\n",
    "    if os.path.exists(LOCAL_DATA_DIR):\n",
    "        print(\"ℹ️ Datos ya descomprimidos en local.\")\n",
    "        return\n",
    "    \n",
    "    if not os.path.exists(DRIVE_DATASET_PATH):\n",
    "        raise FileNotFoundError(f\"❌ No encuentro el dataset en: {DRIVE_DATASET_PATH}. ¡Verifica la ruta en Drive!\")\n",
    "\n",
    "    print(f\"⏳ Descomprimiendo {DRIVE_DATASET_PATH}... (Esto puede tardar unos segundos)\")\n",
    "    shutil.unpack_archive(DRIVE_DATASET_PATH, LOCAL_DATA_DIR)\n",
    "    print(\"✅ Datos listos.\")\n",
    "\n",
    "    # --- FIX DEL YAML ---\n",
    "    # Buscamos y corregimos el data.yaml para que apunte a las rutas de Colab\n",
    "    # Asumimos que dentro del zip hay un data.yaml\n",
    "    potential_yamls = [f for f in os.listdir(LOCAL_DATA_DIR) if f.endswith('.yaml')]\n",
    "    if not potential_yamls:\n",
    "        # Si no está en la raíz, buscamos recursivamente o asumimos nombre estándar\n",
    "        print(\"⚠️ Buscando data.yaml en subcarpetas...\")\n",
    "        for root, dirs, files in os.walk(LOCAL_DATA_DIR):\n",
    "            for file in files:\n",
    "                if file.endswith(\".yaml\"):\n",
    "                     potential_yamls.append(os.path.join(root, file))\n",
    "    \n",
    "    if potential_yamls:\n",
    "        target_yaml = os.path.join(LOCAL_DATA_DIR, potential_yamls[0])\n",
    "        print(f\"🔧 Ajustando rutas en: {target_yaml}\")\n",
    "        \n",
    "        with open(target_yaml, 'r') as f:\n",
    "            data_cfg = yaml.safe_load(f)\n",
    "        \n",
    "        # RUTA MÁGICA: Forzamos rutas absolutas de Colab\n",
    "        data_cfg['path'] = LOCAL_DATA_DIR\n",
    "        data_cfg['train'] = 'train/images'\n",
    "        data_cfg['val'] = 'valid/images'\n",
    "        data_cfg['test'] = 'test/images'\n",
    "        \n",
    "        with open(target_yaml, 'w') as f:\n",
    "            yaml.dump(data_cfg, f)\n",
    "        \n",
    "        return target_yaml\n",
    "    else:\n",
    "        raise FileNotFoundError(\"❌ No encontré ningún archivo .yaml en el dataset descomprimido.\")\n",
    "\n",
    "# Ejecutar preparación\n",
    "final_yaml_path = prepare_data()\n",
    "print(f\"📂 Configuración de datos lista en: {final_yaml_path}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### 4. Ejecución del Entrenamiento\n",
    "Lanzamos el script `main.py` pasando la ruta dinámica del dataset."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Crear carpeta de salida en Drive si no existe\n",
    "os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)\n",
    "\n",
    "# Ejecutar Main\n",
    "# --data: Le pasamos la ruta absoluta del yaml que acabamos de corregir\n",
    "!python main.py --data \"{final_yaml_path}\"\n",
    "\n",
    "# --- GUARDADO DE EMERGENCIA ---\n",
    "# Al terminar, copiamos los resultados de Colab (volátil) a Drive (persistente)\n",
    "print(\"💾 Respaldando resultados a Drive...\")\n",
    "!cp -r runs/ \"{DRIVE_OUTPUT_DIR}/\"\n",
    "print(\"✅ Respaldo completado en 'IPDI/Trabajo Final/Resultados_YOLO'.\")"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.10.12"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 2
}